<a href="https://colab.research.google.com/github/CYRUS-pinto/inscrona/blob/main/Inscrona_Colab_Remote_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inscrona Remote Inference - Google Colab

Run **GLM-OCR** and **Llama 3.2:3B** on Colab's free GPU (T4/A100) and expose via ngrok for your local Inscrona backend.

## Why Colab?
- Free T4 GPU (16GB VRAM) or A100 (40GB) on Colab Pro
- No local GPU needed - runs entirely in cloud
- Falls back when local Ollama is unavailable
- Same models: GLM-OCR (0.9B) + Llama 3.2:3B

## Setup (Run once per session)
1. Open this notebook in Colab
2. Runtime → Change runtime type → GPU (T4)
3. Run all cells below
4. Copy the **ngrok URL** and set in your local `.env`:
   ```bash
   COLAB_INFERENCE_URL=https://xxxx.ngrok-free.app
   ```
5. Your local FastAPI will automatically use Colab when local Ollama fails

In [ ]:
# @title 3️⃣ Start Free Cloudflare Tunnel (No ngrok, zero signup)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, time, re

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

public_url = None
start_t = time.time()
while time.time() - start_t < 15:
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print(f'🌐 Public Tunnel URL: {public_url}')
print(f'📋 Set this in your local .env:')
print(f'   COLAB_INFERENCE_URL={public_url}')


In [ ]:
# @title 2️⃣ Start Ollama & Pull Models
# Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh > /dev/null 2>&1

# Start Ollama in background
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull models (first run takes ~5 min, cached after)
print("📥 Pulling GLM-OCR...")
!ollama pull glm-ocr
print("📥 Pulling Llama 3.2:3B...")
!ollama pull llama3.2:3b

# Verify
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)
print("✅ Models ready")

In [ ]:
# @title 3️⃣ Start Free Cloudflare Tunnel (No ngrok, zero signup)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, time, re

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

public_url = None
start_t = time.time()
while time.time() - start_t < 15:
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print(f'🌐 Public Tunnel URL: {public_url}')
print(f'📋 Set this in your local .env:')
print(f'   COLAB_INFERENCE_URL={public_url}')


In [ ]:
# @title 4️⃣ FastAPI Inference Server
app = FastAPI(title="Inscrona Colab Inference", version="1.0.0")

MAX_LONGEST_EDGE = 2000
OLLAMA_URL = "http://127.0.0.1:11434"

def ollama_generate(model: str, prompt: str, images=None, keep_alive=0, format_json=False, timeout=600):
    payload = {
        "model": model,
        "prompt": prompt,
        "keep_alive": keep_alive,
        "stream": True,
        "options": {"num_predict": 2048, "temperature": 0.1}
    }
    if images:
        payload["images"] = images
    if format_json:
        payload["format"] = "json"
    
    resp = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, stream=True, timeout=timeout)
    resp.raise_for_status()
    full = []
    for line in resp.iter_lines():
        if line:
            chunk = json.loads(line)
            full.append(chunk.get("response", ""))
            if chunk.get("done"):
                break
    return "".join(full)

@app.get("/health")
def health():
    try:
        resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        models = [m["name"] for m in resp.json().get("models", [])]
        return {
            "status": "ok",
            "models": {
                "ocr": {"name": "glm-ocr", "loaded": "glm-ocr:latest" in models},
                "grading": {"name": "llama3.2:3b", "loaded": "llama3.2:3b" in models}
            },
            "backend": "colab",
            "gpu": "T4"  # or A100 on Colab Pro
        }
    except:
        return {"status": "degraded", "backend": "colab"}

@app.post("/grade")
async def grade(file: UploadFile = File(...), rubric: str = Form(default="Rate 0-10 for accuracy, completeness, clarity.")):
    # 1. Read & process image
    raw = await file.read()
    ext = Path(file.filename or "upload.jpg").suffix.lower()
    
    img = Image.open(io.BytesIO(raw))
    if ext in ('.heic', '.heif'):
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=90)
        raw = buf.getvalue()
        img = Image.open(io.BytesIO(raw))
    
    if max(img.size) > MAX_LONGEST_EDGE:
        scale = MAX_LONGEST_EDGE / max(img.size)
        img = img.resize((int(img.width*scale), int(img.height*scale)), Image.LANCZOS)
    
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=90)
    image_b64 = base64.b64encode(buf.getvalue()).decode()

    # 2. OCR with GLM-OCR
    ocr_prompt = "Extract all text from this answer booklet image. Output only the text content, preserving layout."
    ocr_text = ollama_generate("glm-ocr", ocr_prompt, images=[image_b64], keep_alive=0)

    # 3. Grade with Llama 3.2:3B
    grade_prompt = f"""You are an exam grading assistant. Grade the following student answer.

Student Answer:
{ocr_text}

Rubric: {rubric}

Respond ONLY with valid JSON:
{{"marks": <0-10>, "confidence": <0.0-1.0>, "feedback": "<brief assessment>"}}"""
    grade_raw = ollama_generate("llama3.2:3b", grade_prompt, keep_alive=0, format_json=True)

    # 4. Parse JSON
    try:
        cleaned = grade_raw.strip()
        if cleaned.startswith("```"):
            cleaned = "\n".join([l for l in cleaned.split("\n") if not l.startswith("```")])
        parsed = json.loads(cleaned)
        
        marks = parsed.get("marks", 0)
        if isinstance(marks, dict):
            vals = [v for v in marks.values() if isinstance(v, (int, float))]
            marks = round(sum(vals)/len(vals)) if vals else 0
        elif isinstance(marks, str):
            marks = int(marks)
        
        conf = parsed.get("confidence", 0.5)
        if isinstance(conf, str):
            conf = conf.replace("%%", "").strip()
            conf = float(conf)/100.0 if float(conf) > 1 else float(conf)
        
        result = {
            "marks": marks,
            "confidence": conf,
            "feedback": parsed.get("feedback", ""),
            "ocr_text": ocr_text
        }
    except Exception as e:
        raise HTTPException(500, f"JSON parse failed: {e}")

    return JSONResponse(result)

import io
from pathlib import Path
print("✅ FastAPI app created")

In [ ]:
# @title 5️⃣ Run Server (Keep This Cell Running)
# This blocks - keep cell running while you use the API
print("🚀 Starting Colab Inference Server on port 8000...")
print("📡 Endpoints:")
print(f"   GET  {public_url}/health")
print(f"   POST {public_url}/grade")

uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

## 📱 How to Use from Local Inscrona

### 1. Add to local `.env` (in your Inscrona folder):
```bash
COLAB_INFERENCE_URL=https://xxxx.ngrok-free.app
```

### 2. Update local `main.py` to fallback to Colab:
```python
import os
COLAB_URL = os.getenv("COLAB_INFERENCE_URL")

async def grade_with_fallback(image_b64, rubric):
    # Try local first
    try:
        return await local_grade(image_b64, rubric)
    except Exception as e:
        print(f"Local failed: {e}, trying Colab...")
    
    # Fallback to Colab
    if COLAB_URL:
        async with httpx.AsyncClient(timeout=600) as client:
            files = {"file": ("image.jpg", base64.b64decode(image_b64), "image/jpeg")}
            data = {"rubric": rubric}
            resp = await client.post(f"{COLAB_URL}/grade", files=files, data=data)
            resp.raise_for_status()
            return resp.json()
    raise
```

### 3. Test Colab endpoint directly:
```bash
curl -X POST "$COLAB_INFERENCE_URL/grade" \
  -F "file=@test_image.jpg" \
  -F "rubric=Rate 0-10 for accuracy."
```

## ⚠️ Notes
- **Colab free tier**: 12hr session limit, may disconnect on inactivity
- **Keep this notebook tab open** while using
- **ngrok free tier**: 1 session, 40 connections/min
- **First run**: ~5 min to pull models (cached after)
- **Colab Pro**: A100 GPU, longer sessions, priority access

## 🔐 Security
- Add API key auth in production: `Authorization: Bearer <token>`
- Restrict ngrok: `ngrok http 8000 --basic-auth "user:pass"`
- Or use Cloudflare Tunnel for permanent domain